# Step 5: Carbon Stock Accounting with ±95% CI Uncertainty

Builds a species map (50% Sitka Spruce / 30% Lodgepole Pine / 20% Native Mixed — replace with the Step-4 output in production) and runs the Tier-2 biomass matrix from `src/carbon_accounting.py`. Band 1 = biomass (t/ha), Band 2 = ±95% CI.

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.transform import from_bounds

sys.path.append(os.path.abspath('../src'))
from carbon_accounting import calculate_tier2_biomass

os.makedirs('../data/output', exist_ok=True)
rng = np.random.default_rng(42)
species = rng.choice([1, 2, 3], size=(200, 200), p=[0.5, 0.3, 0.2]).astype('uint8')
profile = {"driver": "GTiff", "height": 200, "width": 200, "count": 1, "dtype": "uint8",
           "crs": "EPSG:25829",
           "transform": from_bounds(598000, 5862000, 604000, 5868000, 200, 200)}
with rasterio.open('../data/output/species_map.tif', 'w', **profile) as dst:
    dst.write(species, 1)

calculate_tier2_biomass('../data/output/species_map.tif', '../data/output/04_carbon_stock.tif')

with rasterio.open('../data/output/04_carbon_stock.tif') as src:
    bio, unc = src.read(1), src.read(2)

names = {1: 'Sitka Spruce', 2: 'Lodgepole Pine', 3: 'Native Mixed'}
for sp in (1, 2, 3):
    m = species == sp
    print(f"{names[sp]:<15}: {int(m.sum()):>5} px | mean biomass {bio[m].mean():6.1f} t/ha | 95% CI ±{unc[m].mean():5.1f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
im0 = axes[0].imshow(bio, cmap='YlOrRd')
axes[0].set_title('Tier-2 biomass (t/ha)')
fig.colorbar(im0, ax=axes[0], fraction=0.04)
im1 = axes[1].imshow(unc, cmap='Purples')
axes[1].set_title('95% CI uncertainty')
fig.colorbar(im1, ax=axes[1], fraction=0.04)
for ax in axes:
    ax.axis('off')
plt.show()